# 02 · Agregación por vehículo y día (exploración)

Pasa de millones de viajes y mensajes de la ECU a una fila por vehículo y día (tabla `diario`), incluyendo las variables
del **mecanismo de regeneración** medidas viaje a viaje (viajes con el filtro cargado, cortos con el filtro cargado,
regeneraciones completas e interrumpidas) y el **ralentí**.

**Entrada y salida:** `salidas/fic.duckdb`. Correr después de `01_organizar_datos`.

In [ ]:
import os
import duckdb
import pandas as pd

# Rutas relativas a notebooks/exploracion/ (los CSV de Ford quedan en Datasets/, no versionado)
DATOS   = "../../Datasets/"
STATIC  = DATOS + "Static/"
TRIPS   = DATOS + "TripSummary/"
DYN     = DATOS + "Dynamic/"
SALIDAS = "salidas/"          # fic.duckdb, dataset.parquet, etc. (no versionado)
os.makedirs(SALIDAS, exist_ok=True)

# DuckDB siempre con límite de recursos (regla del proyecto: sin límite puede congelar una máquina de 16 GB)
con = duckdb.connect(SALIDAS + "fic.duckdb", config={"memory_limit": "4GB", "threads": 4})
con.sql("SET TimeZone='UTC'")

In [16]:
con.sql("""
create or replace table diario_trips as
select VehicleCode,
       TripDatetimeStart::date                                   as dia,
       count(*)                                                  as viajes,
       sum(km)                                                   as km,
       sum(minutos)                                              as minutos,
       avg((km < 5)::int)                                        as pct_cortos,
       avg((EngineTemperatureMax < 80)::int)                     as pct_motor_frio,
       avg(KilometerPerHour)                                     as vel_media,
       max(AirRegenerationEnd)                                   as acum_max_trip,
       sum((AirFilterEnd in ('Air Filter Full', 'Air Filter Overloaded',
                             'Air Filter Over Limit', 'Air Filter At Limit'))::int) as viajes_lleno,
       min(EngineOilLifePCEnd)                                   as aceite,
       avg(AirTemperatureAvg)                                    as temp_aire,
       100 * sum(greatest(FuelLvlStartPc - FuelLvlEndPc, 0)) / nullif(sum(km), 0) as consumo_pct_100km,
       count(*) filter (where AirRegenerationStart >= 90)                      as viajes_filtro_cargado,
       count(*) filter (where AirRegenerationStart >= 90 and minutos < 20)     as cortos_filtro_cargado,
       count(*) filter (where AirFilterEnd like 'Cleaning Automatically%')     as regen_interrumpidas,
       count(*) filter (where AirRegenerationEnd <= AirRegenerationStart - 30) as regen_completas
from trips_limpios
group by all
""")
con.sql("select count(*) from diario_trips").df()

,count_star()
0,247686


In [17]:
con.sql("""
create or replace table diario_dyn as
select d.VehicleCode,
       d.eventTimestamp::date                                        as dia,
       count(*)                                                      as registros,
       max(d.Acumulation)                                            as acum_max,
       avg(d.Acumulation)                                            as acum_media,
       avg((d.Message in ('Air Filter Full', 'Air Filter Overloaded',
                          'Air Filter Over Limit', 'Air Filter At Limit'))::int) as pct_lleno,
       count(*) filter (where d.Regenerations = 'Regeneration')      as regeneraciones,
       avg(try_cast(d.DistanceBetweenRegenerations as double))
           filter (where try_cast(d.DistanceBetweenRegenerations as double) between 1 and 5000) as dist_regen,
       count(*) filter (where d.Message like 'Cleaning Automatically%') as limpiezas_auto,
       count(*) filter (where d.Message like 'Stopped%')                as limpiezas_cortadas
from dynamic d
join vehiculos v using (VehicleCode)
where d.eventTimestamp::date >= v.venta_date
group by all
""")

In [18]:
con.sql("""
create or replace table diario_ralenti as
select VehicleCode, dia,
       sum(minutos)                         as min_ralenti,
       count(*) filter (where minutos >= 5) as ralentis_largos
from ralenti
group by all
""")

In [19]:
con.sql("""
create or replace table diario as
select coalesce(t.VehicleCode, d.VehicleCode) as VehicleCode,
       coalesce(t.dia, d.dia)                 as dia,
       t.* exclude (VehicleCode, dia),
       d.* exclude (VehicleCode, dia)
from diario_trips t
full outer join diario_dyn d
  on t.VehicleCode = d.VehicleCode and t.dia = d.dia
""")

In [20]:
con.sql("""
create or replace table diario as
select d.*,
       coalesce(r.min_ralenti, 0)     as min_ralenti,
       coalesce(r.ralentis_largos, 0) as ralentis_largos,
       v.failed, v.Engine, v.ModelSeries, v.pais, v.ciudad,
       d.dia - v.prod_date  as edad_dias,
       v.event_date - d.dia as dias_hasta_evento
from diario d
join vehiculos v using (VehicleCode)
left join diario_ralenti r using (VehicleCode, dia)
""")
con.sql("""select failed, count(*) as filas, count(distinct VehicleCode) as vehiculos
           from diario group by failed""").df()

,failed,filas,vehiculos
0,1,101564,277
1,0,165622,709


In [21]:
con.sql("""
select (floor(dias_hasta_evento / 30) * 30)::int as dias_antes,
       count(*)                                  as dias,
       round(avg(pct_lleno), 3)                  as pct_lleno,
       round(avg(pct_cortos), 3)                 as pct_cortos,
       round(avg(consumo_pct_100km), 2)          as consumo,
       round(avg(acum_media), 1)                 as acum_media
from diario
where failed = 1 and dias_hasta_evento between -60 and 179
group by 1
order by 1 desc
""").df()

,dias_antes,dias,pct_lleno,pct_cortos,consumo,acum_media
0,150,4513,0.031,0.434,16.94,41.4
1,120,5088,0.034,0.444,16.86,41.6
2,90,5549,0.035,0.437,16.88,40.1
3,60,5964,0.041,0.456,18.40,40.3
4,30,5847,0.042,0.451,17.17,41.3
5,0,5784,0.040,0.449,17.50,38.5
6,-30,6196,0.031,0.438,17.30,42.5
7,-60,5384,0.031,0.448,17.81,42.3


In [22]:
con.close()